# Predicting `arrears_next_month`
Protocol: `protocol.yaml`. Temporal holdout from `holdout.csv`, walk-forward CV (5 splits) on training rows only, model chosen by mean CV ROC AUC, refit on all training rows and scored once on the holdout.

**Feature decision:** `final_notice_sent` is excluded. Per `columns.csv` it is recorded in the billing cycle *after* the billing month, i.e. after/around the outcome, so it is not known at prediction time (and it almost determines the target: 87% vs 1.4% arrears). `meter_ref` and `billing_month` are excluded per the protocol.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv('data.csv')
assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='42accdc3f8c6ae3514896d73516d98c54e4a27c5422d4fe88e5c4fe1b00a14f7'
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest()=='903bb064736772428ae27c90c6c03ac7f9c28c7eb54a50c70b7601b731258ba6'
assert len(df)==20939
hold_idx = pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(hold_idx)==7536
is_hold = np.zeros(len(df),bool); is_hold[hold_idx]=True
train = df[~is_hold]; hold = df[is_hold]
print(len(train), len(hold), train.billing_month.max(), hold.billing_month.min())
assert train.billing_month.max() < hold.billing_month.min()

In [ ]:
TARGET='arrears_next_month'
CAT=['tariff']; NUM=['occupants','kwh','bill_eur','late_payments_12m']
FEATURES=CAT+NUM
print(train[TARGET].mean(), hold[TARGET].mean())

## Candidates (named before any scoring)

In [ ]:
def prep():
    return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),CAT),('n',StandardScaler(),NUM)])
candidates = {
 'baseline_prior': Pipeline([('p',prep()),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C1': Pipeline([('p',prep()),('m',LogisticRegression(C=1.0,max_iter=1000))]),
 'logreg_C0.1': Pipeline([('p',prep()),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'random_forest': Pipeline([('p',prep()),('m',RandomForestClassifier(n_estimators=300,min_samples_leaf=20,n_jobs=-1,random_state=0))]),
 'hist_gb': Pipeline([('p',prep()),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))]),
}
list(candidates)

## Walk-forward CV on training rows (expanding window over months, 5 splits)

In [ ]:
months = np.sort(train.billing_month.unique())
chunks = np.array_split(months, 6)   # chunk 0 = initial train, chunks 1..5 = validation folds
folds=[]
for k in range(1,6):
    tr_m = np.concatenate(chunks[:k]); va_m = chunks[k]
    tr = np.where(train.billing_month.isin(tr_m))[0]; va = np.where(train.billing_month.isin(va_m))[0]
    folds.append((tr,va))
    print(k, len(tr), len(va), va_m[0], va_m[-1])
X=train[FEATURES]; y=train[TARGET].to_numpy()
cv_scores={}
for name,est in candidates.items():
    s=[]
    for tr,va in folds:
        m=clone(est).fit(X.iloc[tr],y[tr])   # preprocessing fitted inside each fold
        s.append(float(roc_auc_score(y[va], m.predict_proba(X.iloc[va])[:,1])))
    cv_scores[name]=s
cv_tab=pd.DataFrame(cv_scores).T; cv_tab['mean']=cv_tab.mean(axis=1); cv_tab

In [ ]:
chosen = max(cv_scores, key=lambda k: np.mean(cv_scores[k]))
print('chosen:', chosen, np.mean(cv_scores[chosen]))

## Refit on all training rows, score once on holdout

In [ ]:
final = clone(candidates[chosen]).fit(X,y)
holdout_auc = float(roc_auc_score(hold[TARGET], final.predict_proba(hold[FEATURES])[:,1]))
print('HOLDOUT roc_auc', holdout_auc)

In [ ]:
train_pos = np.where(~is_hold)[0]
ev = {
 'features': FEATURES,
 'candidates': list(candidates),
 'cv_folds': [[[int(i) for i in train_pos[tr]], [int(i) for i in train_pos[va]]] for tr,va in folds],
 'cv_scores': cv_scores,
 'chosen': chosen,
 'holdout_rows': [int(i) for i in np.sort(hold_idx)],
 'holdout_scores': {chosen: holdout_auc},
}
json.dump(ev, open('evidence.json','w'))

## Conclusion
See the CV table and holdout ROC AUC above. `final_notice_sent` is not available at prediction time, so the figures here are the honest estimate of how predictable next-month arrears are from the account's own billing history.